In [1]:
import sys
import os
import urllib3

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)

Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [2]:
import pandas as pd
from datetime import datetime, timedelta
import pytz

# Define time period
# Calculate the start date (2 days ago) at 00:00:00 UTC
start_date = (datetime.now(pytz.UTC) - timedelta(days=2)).date()

# Format as 'YYYY-MM-DDT00:00:00Z'
start = f"{start_date}T00:00:00Z"

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        observed_src = observed_src[observed_src['lastObserved'] >= start]
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")

display(observed_src)


Querying owner: HTOC Org

Retrieved 812 unique observed indicators.


,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,source,hostName,dnsActive,whoisActive,text,address,url,sha256,Block,indicator
0,4553603,2024-03-29T13:13:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-20T16:36:59Z,3.0,0.0,Indicators uploaded to ThreatConnect from CMS'...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.246.8.131
1,6755399447111235,2025-05-14T17:44:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-20T16:32:00Z,3.0,54.0,FBI Email Alert May 14 2025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.152.52.216
2,5629499536037723,2025-04-15T17:38:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-20T16:32:00Z,5.0,70.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.200.148.242
3,6755399481063931,2025-11-17T16:07:55Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-20T16:27:00Z,3.0,80.0,TASK0931507,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.142.193.171
4,5629499566213979,2025-09-04T14:20:28Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-20T15:41:59Z,3.0,69.0,INC9223440,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.159.71.249
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9821,5629499556005878,2025-06-30T12:21:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-17T08:36:59Z,3.0,60.0,RITM0594014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.114
9822,5629499556005877,2025-06-30T12:21:42Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-17T08:36:59Z,3.0,60.0,RITM0594014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.182
9823,5629499556005870,2025-06-30T12:21:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-17T08:36:59Z,3.0,60.0,RITM0594014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.62.156.49
9824,5629499556005850,2025-06-30T12:21:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-17T08:36:59Z,3.0,60.0,RITM0594014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.62.156.47


In [3]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251120.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251119.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251118.csv']


C:\Users\jaskew\AppData\Local\Temp\ipykernel_22728\564055639.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Loaded data from 3 files.


In [8]:
import pandas as pd
from datetime import timedelta
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION & SETUP
# ═══════════════════════════════════════════════════════════════════════════════

# Time cutoffs
cutoff = pd.Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=180)
cutoff_naive = cutoff.tz_convert(None)

# Required columns validation
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [c for c in required_columns if c not in observed_data_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def process_tag_row(row, observed_src):
    """Process a single row from observed_src to extract and filter tags."""
    tags_data = row.get('tags.data')
    if not isinstance(tags_data, list):
        return None

    # Normalize the tags for the current row
    tags_df = pd.json_normalize(tags_data)

    # Ensure string and apply VA rename before filtering
    tags_df['name'] = (
        tags_df['name']
        .astype(str)
        .str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)
    )

    # Skip if all associated groups are Adversary
    if 'associatedGroups.data' in observed_src.columns:
        ag_data = row.get('associatedGroups.data')
        if isinstance(ag_data, list) and len(ag_data) > 0:
            groups_df = pd.json_normalize(ag_data)
            if 'type' in groups_df.columns and set(groups_df['type']) == {'Adversary'}:
                return None

    # All tag names (for all_tags)
    all_tags_list = tags_df['name'].tolist()

    # Filter for API tags
    api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)].copy()
    if api_tags.empty:
        return None

    # Add metadata columns
    meta_cols = [
        'summary', 'observations', 'description', 'type',
        'dateAdded', 'lastModified', 'lastObserved', 'webLink',
        'rating', 'confidence', 'id', 'associatedGroups.data'
    ]
    for col in meta_cols:
        api_tags[col] = [row.get(col)] * len(api_tags)

    # Attach all tags list
    if len(api_tags) > 0:
        api_tags['all_tags'] = [all_tags_list] * len(api_tags)

    return api_tags

def map_observed_dates(filtered_tags, observed_data_df):
    """Map observed dates from observed_data_df to filtered_tags."""
    if filtered_tags.empty:
        return filtered_tags
    
    # Clean name -> match OpDiv (remove trailing ' Splunk API')
    filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)
    filtered_tags['observed_date'] = pd.NaT
    
    # Ensure observed_data_df obs_date is datetime (naive)
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')

    for idx, r in filtered_tags.iterrows():
        summary = r.get('summary')
        cleaned_name = r.get('cleaned_name')
        if pd.isna(summary) or pd.isna(cleaned_name):
            continue
        match = observed_data_df[
            (observed_data_df['indicator'] == summary) &
            (observed_data_df['OpDiv'] == cleaned_name)
        ]
        if not match.empty:
            filtered_tags.at[idx, 'observed_date'] = match['obs_date'].iloc[0]

    filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')
    
    # Drop helper column
    filtered_tags.drop(columns=['cleaned_name'], inplace=True, errors='ignore')
    
    return filtered_tags

def get_multi_partner_indicators(observed_data_df, cutoff_naive):
    """Get indicators that have multiple partners from observed_data_df."""
    if observed_data_df.empty:
        return pd.DataFrame()
    
    # Ensure obs_date is datetime
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')
    
    # Filter by recent dates (last 3 days)
    recent_obs = observed_data_df[
        observed_data_df['obs_date'] >= cutoff_naive - timedelta(days=3)
    ].copy()
    
    if recent_obs.empty:
        return pd.DataFrame()
    
    # Group by indicator and count unique OpDiv (partners)
    partner_counts = (
        recent_obs.groupby('indicator')['OpDiv']
        .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
        .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
    )
    
    # Keep only indicators with multiple partners
    multi_partner_indicators = partner_counts[partner_counts['partner_count'] >= 2].copy()
    
    return multi_partner_indicators

def apply_filters_and_grouping(filtered_tags, cutoff, date_added_cutoff, cutoff_naive, multi_partner_indicators):
    """Apply time filters, partner grouping, and other filtering criteria."""
    if filtered_tags.empty:
        return pd.DataFrame()
    
    # Time-based filters
    # Last 24h by lastObserved (aware) - but we'll use the 2-day filter below
    # recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=24)].copy()
    
    # Last 2 days by observed_date (naive)
    recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()

    if recent_tags.empty:
        return recent_tags
    
    # Filter to only include indicators that have multiple partners in observed_data_df
    if not multi_partner_indicators.empty:
        recent_tags = recent_tags[
            recent_tags['summary'].isin(multi_partner_indicators['indicator'])
        ].copy()
    
    if recent_tags.empty:
        return recent_tags
    
    # Partner extraction and grouping from ThreatConnect tags (as fallback)
    recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

    partner_groups = (
        recent_tags.groupby('summary')['partner']
        .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
        .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
    )

    recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')
    
    # Merge with multi_partner_indicators to get partners from observed_data_df
    if not multi_partner_indicators.empty:
        recent_tags = recent_tags.merge(
            multi_partner_indicators[['indicator', 'partners', 'partner_count']],
            left_on='summary',
            right_on='indicator',
            how='left',
            suffixes=('', '_from_obs')
        )
        # Use partners from observed_data_df if available, otherwise use from tags
        recent_tags['partners'] = recent_tags['partners_from_obs'].fillna(recent_tags['partners'])
        recent_tags['partner_count'] = recent_tags['partner_count_from_obs'].fillna(recent_tags['partner_count'])
        recent_tags = recent_tags.drop(columns=['indicator', 'partners_from_obs', 'partner_count_from_obs'], errors='ignore')

    # Additional filters
    recent_tags = recent_tags[recent_tags['partner_count'] >= 2]
    recent_tags = recent_tags[recent_tags['observations'] < 15000]
    recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]

    # Deduplicate by summary
    recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

    # Drop unneeded columns if present
    cols_to_drop = [
        'techniqueId', 'tactics.data', 'tactics.count',
        'platforms.data', 'platforms.count', 'partner', 'name'
    ]
    recent_tags = recent_tags.drop(columns=[c for c in cols_to_drop if c in recent_tags.columns], errors='ignore')

    # Remove rows where all_tags contains unwanted markers (keep htoc_wl filter, but not I&W)
    if 'all_tags' in recent_tags.columns:
        recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'htoc_wl' in x)]
    
    # Add partners from tags at the end (after all filtering)
    # Re-extract partners from tags for the final filtered dataset
    if not recent_tags.empty and 'all_tags' in recent_tags.columns:
        def extract_partners_from_tags(all_tags_list):
            """Extract partner names from tags that contain 'API'."""
            if not isinstance(all_tags_list, list):
                return []
            api_partners = []
            for tag in all_tags_list:
                if isinstance(tag, str) and 'API' in tag:
                    # Extract partner name (remove ' Splunk API' suffix)
                    partner = tag.replace(' Splunk API', '').replace('VA CSOC CTS Splunk', 'VA').strip()
                    if partner:
                        api_partners.append(partner)
            return sorted(set(api_partners))
        
        # Extract partners from tags for each indicator
        tag_partners_series = recent_tags['all_tags'].apply(extract_partners_from_tags)
        
        # Combine with existing partners from observed_data_df
        def combine_partners(row_idx):
            """Combine partners from observed_data_df and tags for a specific row."""
            obs_partners = recent_tags.loc[row_idx, 'partners']
            tag_partners_list = tag_partners_series.loc[row_idx]
            
            combined = set()
            # Add partners from observed_data_df
            if pd.notna(obs_partners) and obs_partners:
                for p in str(obs_partners).split(', '):
                    if p.strip():
                        combined.add(p.strip())
            # Add partners from tags
            if tag_partners_list:
                for p in tag_partners_list:
                    if p:
                        combined.add(p)
            return ', '.join(sorted(combined)) if combined else obs_partners
        
        recent_tags['partners'] = recent_tags.index.to_series().apply(combine_partners)
        
        # Update partner_count based on combined partners
        recent_tags['partner_count'] = recent_tags['partners'].apply(
            lambda x: len([p for p in str(x).split(', ') if p.strip()]) if pd.notna(x) and x else 0
        )

    return recent_tags

def extract_group_ids(recent_tags):
    """Extract group IDs from associatedGroups.data."""
    if 'associatedGroups.data' in recent_tags.columns:
        recent_tags['group_id'] = recent_tags['associatedGroups.data'].apply(
            lambda x: x[0]['id'] if isinstance(x, list) and len(x) > 0 and 'id' in x[0] else None
        )
        # Convert group_id to string type and remove trailing decimals if it exists
        if 'group_id' in recent_tags.columns:
            recent_tags['group_id'] = recent_tags['group_id'].apply(
                lambda x: str(int(float(x))) if pd.notna(x) and str(x) != 'None' else x
            ).astype(str)
    return recent_tags

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN PROCESSING PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

print("Starting tag processing pipeline...")

# Step 1: Process tags from observed_src
print("Processing tags from observed_src...")
all_filtered = []

for _, row in observed_src.iterrows():
    processed_tags = process_tag_row(row, observed_src)
    if processed_tags is not None:
        all_filtered.append(processed_tags)

# Step 2: Create filtered_tags DataFrame
print("Creating filtered_tags DataFrame...")
filtered_tags = pd.concat(all_filtered, ignore_index=True) if all_filtered else pd.DataFrame()

if not filtered_tags.empty:
    # Ensure datetime columns
    filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce', utc=True)
    filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce', utc=True)
    print(f"Created filtered_tags with {len(filtered_tags)} rows")
else:
    print("No filtered tags found")

# Step 3: Map observed dates
print("Mapping observed dates...")
filtered_tags = map_observed_dates(filtered_tags, observed_data_df)

# Step 3.5: Get indicators with multiple partners from observed_data_df
print("Identifying indicators with multiple partners from observed_data_df...")
multi_partner_indicators = get_multi_partner_indicators(observed_data_df, cutoff_naive)
print(f"Found {len(multi_partner_indicators)} indicators with multiple partners")

# Step 4: Apply filters and grouping
print("Applying filters and partner grouping...")
recent_tags = apply_filters_and_grouping(filtered_tags, cutoff, date_added_cutoff, cutoff_naive, multi_partner_indicators)

# Step 5: Extract group IDs
print("Extracting group IDs...")
recent_tags = extract_group_ids(recent_tags)

# Step 6: Add I&W column indicating if indicator has been tagged as 'I&W' (used in previous report)
if 'all_tags' in recent_tags.columns:
    recent_tags['I&W'] = recent_tags['all_tags'].apply(
        lambda x: 'Yes' if isinstance(x, list) and 'I&W' in x else 'No'
    )
else:
    recent_tags['I&W'] = 'No'

# Move I&W column to the very end
cols = [col for col in recent_tags.columns if col != 'I&W'] + ['I&W']
recent_tags = recent_tags[cols]

# Final summary
print(f"Processing complete! Final dataset shape: {recent_tags.shape}")
if not recent_tags.empty:
    print(f"Partners represented: {recent_tags['partners'].nunique()} unique partner combinations")
    print(f"Date range: {recent_tags['observed_date'].min()} to {recent_tags['observed_date'].max()}")

recent_tags.head(20)

Starting tag processing pipeline...
Processing tags from observed_src...
Creating filtered_tags DataFrame...
Created filtered_tags with 5800 rows
Mapping observed dates...
Identifying indicators with multiple partners from observed_data_df...
Found 699 indicators with multiple partners
Applying filters and partner grouping...
Extracting group IDs...
Processing complete! Final dataset shape: (25, 19)
Partners represented: 18 unique partner combinations
Date range: 2025-11-19 00:00:00 to 2025-11-20 00:00:00


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id,I&W
233,5629499574089565,2025-11-13T07:08:13Z,Key Findings\nSilent Push Threat Analysts have...,208.115.228.234,32,Address,2025-10-21 11:33:56+00:00,2025-11-20T13:56:59Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,96.0,"[{'id': 6755399470002411, 'dateAdded': '2025-1...","[OtterCookie, Contagious Interview, Illicit IT...",2025-11-19,3,"NIH, OS, VA",6755399470002411,Yes
234,5629499574089567,2025-11-13T07:08:13Z,Key Findings\nSilent Push Threat Analysts have...,23.106.161.1,17,Address,2025-10-21 11:33:56+00:00,2025-11-20T13:56:41Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,96.0,"[{'id': 6755399470002411, 'dateAdded': '2025-1...","[OtterCookie, Contagious Interview, Illicit IT...",2025-11-19,3,"DHA, NIH, VA",6755399470002411,Yes
728,5629499542029727,2025-11-12T16:27:29Z,INC9072336,187.141.210.92,12481,Address,2025-05-27 13:19:06+00:00,2025-11-20T09:37:00Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,55.0,NaN,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-11-20,6,"CMS, FDA, HHS, HRSA, OS, VA",nan,No
869,6755399459075899,2025-11-13T05:43:04Z,MDE alert was triggered - TI Map IP Entity to ...,185.220.101.30,6116,Address,2025-06-24 16:51:22+00:00,2025-11-20T09:05:28Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,64.0,"[{'id': 5629499557001566, 'dateAdded': '2025-1...","[TOR Node, Phishing, Drive-by Compromise, smis...",2025-11-20,7,"CMS, FDA, HHS, HRSA, IHS, NIH, OS",5629499557001566,Yes
1148,6755399469000430,2025-11-12T16:27:29Z,TASK0912447,45.138.16.69,2849,Address,2025-08-22 14:49:34+00:00,2025-11-20T08:29:38Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,68.0,NaN,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-11-20,5,"CMS, FDA, HRSA, OS, VA",nan,No
1152,5629499555060670,2025-11-13T13:28:14Z,RITM0589984,192.42.116.181,8660,Address,2025-06-16 17:42:13+00:00,2025-11-20T04:06:59Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,58.0,NaN,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-11-20,6,"CMS, FDA, HHS, HRSA, OS, VA",nan,No
1153,6755399459033760,2025-11-13T05:43:04Z,RITM0589984,109.70.100.2,7431,Address,2025-06-16 17:42:43+00:00,2025-11-20T03:37:00Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,58.0,NaN,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-11-20,7,"CMS, FDA, HHS, HRSA, IHS, OS, VA",nan,No
1154,5629499565000772,2025-11-13T05:43:04Z,TASK0912447,185.241.208.71,2573,Address,2025-08-22 14:49:15+00:00,2025-11-19T21:06:59Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,68.0,NaN,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-11-19,4,"CMS, FDA, HRSA, OS",nan,No
1163,6755399460008019,2025-11-13T07:08:20Z,"Details\nIn late June 2025, Iran-aligned hackt...",134.199.159.23,7642,Address,2025-07-02 12:05:33+00:00,2025-11-19T17:31:59Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,53.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-11-20,8,"CMS, DHA, FDA, HHS, HRSA, NIH, OS, VA",5629499544001057,No
1414,5629499574089571,2025-11-13T07:08:20Z,Key Findings\nSilent Push Threat Analysts have...,45.86.208.162,807,Address,2025-10-21 11:33:56+00:00,2025-11-19T08:52:12Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,96.0,"[{'id': 6755399468000794, 'dateAdded': '2025-1...","[Famous Chollima, OtterCookie, Contagious Inte...",2025-11-20,6,"CMS, DHA, FDA, NIH, OS, VA",6755399468000794,Yes


In [9]:
import pandas as pd

# Extract unique indicators from recent_tags
indicators = recent_tags['summary'].unique()

# Initialize a list to store attribute data
attributes_data = []

# Track indicators with no attributes
indicators_with_no_attributes = []

# Iterate over unique indicators
for indicator in indicators:
    try:
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            data = response.json().get('data', {})
            attributes = data.get('attributes', {}).get('data', [])

            if not attributes:
                print(f"No attributes found for indicator: {indicator}")
                # Track indicators with no attributes
                indicators_with_no_attributes.append(indicator)
            else:
                for attr in attributes:
                    attr.update({
                        'id': data.get('id'),
                        'summary': data.get('summary'),
                        'type': data.get('type'),
                        'ownerName': data.get('ownerName')
                    })
                    attributes_data.append(attr)
        # Suppress non-JSON and known 400 error responses
    except Exception as e:
        # Suppress error output, including known 400 error
        if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
            continue
        if "Status Code: 400" in str(e):
            continue
        pass

# Create attributes 
attributes_observed_src = pd.json_normalize(attributes_data)

# Un-nest 'createdBy' and filter out 'SOAR' entries
if not attributes_observed_src.empty and attributes_observed_src['createdBy.lastName'].notnull().any():
    attributes_observed_src = attributes_observed_src[attributes_observed_src['createdBy.lastName'] != 'SOAR']

# Drop duplicates based on 'id' to keep unique attribute records
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

# Filter recent_tags for indicators that had attributes
filtered_with_attrs = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])]

# Filter recent_tags for indicators that had no attributes
no_attrs_df = recent_tags[recent_tags['summary'].isin(indicators_with_no_attributes)]

# Combine both and remove duplicates based on 'summary'
filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)
display(filtered_recent_tags)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,associatedGroups.data,all_tags,observed_date,partner_count,partners,group_id,I&W
0,5629499574089565,2025-11-13T07:08:13Z,Key Findings\nSilent Push Threat Analysts have...,208.115.228.234,32,Address,2025-10-21 11:33:56+00:00,2025-11-20T13:56:59Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,96.0,"[{'id': 6755399470002411, 'dateAdded': '2025-1...","[OtterCookie, Contagious Interview, Illicit IT...",2025-11-19,3,"NIH, OS, VA",6755399470002411,Yes
1,5629499574089567,2025-11-13T07:08:13Z,Key Findings\nSilent Push Threat Analysts have...,23.106.161.1,17,Address,2025-10-21 11:33:56+00:00,2025-11-20T13:56:41Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,96.0,"[{'id': 6755399470002411, 'dateAdded': '2025-1...","[OtterCookie, Contagious Interview, Illicit IT...",2025-11-19,3,"DHA, NIH, VA",6755399470002411,Yes
2,6755399459075899,2025-11-13T05:43:04Z,MDE alert was triggered - TI Map IP Entity to ...,185.220.101.30,6116,Address,2025-06-24 16:51:22+00:00,2025-11-20T09:05:28Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,64.0,"[{'id': 5629499557001566, 'dateAdded': '2025-1...","[TOR Node, Phishing, Drive-by Compromise, smis...",2025-11-20,7,"CMS, FDA, HHS, HRSA, IHS, NIH, OS",5629499557001566,Yes
3,6755399460008019,2025-11-13T07:08:20Z,"Details\nIn late June 2025, Iran-aligned hackt...",134.199.159.23,7642,Address,2025-07-02 12:05:33+00:00,2025-11-19T17:31:59Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,53.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-11-20,8,"CMS, DHA, FDA, HHS, HRSA, NIH, OS, VA",5629499544001057,No
4,5629499574089571,2025-11-13T07:08:20Z,Key Findings\nSilent Push Threat Analysts have...,45.86.208.162,807,Address,2025-10-21 11:33:56+00:00,2025-11-19T08:52:12Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,96.0,"[{'id': 6755399468000794, 'dateAdded': '2025-1...","[Famous Chollima, OtterCookie, Contagious Inte...",2025-11-20,6,"CMS, DHA, FDA, NIH, OS, VA",6755399468000794,Yes
5,5629499574089564,2025-11-12T00:25:06Z,Key Findings\nSilent Push Threat Analysts have...,204.188.233.66,511,Address,2025-10-21 11:33:56+00:00,2025-11-19T08:51:59Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,5.0,96.0,"[{'id': 6755399468000794, 'dateAdded': '2025-1...","[OtterCookie, Contagious Interview, Illicit IT...",2025-11-20,7,"CMS, DHA, FDA, HHS, IHS, OS, VA",6755399468000794,Yes
6,6755399460008397,2025-11-13T07:08:20Z,"In late June 2025, Iran-aligned hacktivist gro...",190.242.157.230,37,Address,2025-07-02 12:05:37+00:00,2025-11-18T08:46:39Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,54.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-11-20,4,"DHA, HRSA, NIH, VA",5629499544001057,No
7,6755399460008419,2025-11-13T13:28:14Z,"In late June 2025, Iran-aligned hacktivist gro...",201.254.226.17,41,Address,2025-07-02 12:05:37+00:00,2025-11-18T08:46:21Z,2025-11-20 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,54.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-11-20,5,"DHA, HHS, HRSA, OS, VA",5629499544001057,No
8,6755399460008180,2025-11-13T07:08:20Z,"In late June 2025, Iran-aligned hacktivist gro...",45.239.49.52,212,Address,2025-07-02 12:05:35+00:00,2025-11-18T08:46:21Z,2025-11-19 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,54.0,"[{'id': 5629499544001057, 'dateAdded': '2025-0...","[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-11-19,6,"CMS, DHA, HHS, HRSA, NIH, VA",5629499544001057,No
9,6755399460008448,2025-11-13T07:08:20Z,"In late June 2025, Iran-aligned hackti

In [28]:
# Save all columns from filtered_recent_tags to Excel with clickable hyperlinks for 'webLink'
output_dir = r"Z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Spreadsheet"
os.makedirs(output_dir, exist_ok=True)
excel_path = os.path.join(
    output_dir,
    f"I&W_indicators_full_{pd.Timestamp.now().strftime('%Y%m%d')}.xlsx"
)

try:
    import openpyxl
    from openpyxl.styles import Font

    # Prepare data for Excel - convert all complex data types to strings
    excel_data = filtered_recent_tags.copy()
    
    # Convert timezone-aware datetime columns to timezone-naive
    for col in excel_data.columns:
        if excel_data[col].dtype == 'datetime64[ns, UTC]' or (excel_data[col].dtype == 'object' and 
                                                             excel_data[col].apply(lambda x: hasattr(x, 'tzinfo') and x.tzinfo is not None).any()):
            excel_data[col] = pd.to_datetime(excel_data[col], errors='coerce').dt.tz_convert(None)
    
    # Convert complex data types to strings
    for col in excel_data.columns:
        if excel_data[col].dtype == 'object':  # Check if column might contain complex objects
            excel_data[col] = excel_data[col].apply(
                lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x) if x is not None else ''
            )

    # Create a new workbook and worksheet
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "I&W Indicators Full"

    # Write headers
    for col_idx, col_name in enumerate(excel_data.columns, 1):
        ws.cell(row=1, column=col_idx, value=col_name)

    # Write data rows
    for row_idx, row in enumerate(excel_data.itertuples(index=False), 2):
        for col_idx, value in enumerate(row, 1):
            col_name = excel_data.columns[col_idx - 1]
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            # Make 'webLink' column clickable
            if col_name == 'webLink' and pd.notna(value) and value != '':
                cell.hyperlink = value
                cell.font = Font(color="0563C1", underline="single")

    # Auto-adjust column widths
    for column_cells in ws.columns:
        max_length = 0
        column_letter = column_cells[0].column_letter
        for cell in column_cells:
            try:
                if cell.value is not None and len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except Exception:
                pass
        ws.column_dimensions[column_letter].width = min(max_length + 2, 50)

    wb.save(excel_path)
    print(f"Saved all filtered_recent_tags data to Excel: {excel_path}")

except ImportError:
    print("openpyxl not available. Install with: pip install openpyxl")
    print("Excel file with hyperlinks not created.")
except Exception as e:
    print(f"Error creating Excel file: {e}")

Saved all filtered_recent_tags data to Excel: Z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Spreadsheet\I&W_indicators_full_20251008.xlsx
